Este módulo construye un índice ponderado por capitalización para 20 acciones del S&P 500, calcula el índice de concentración de Herfindahl-Hirschman (HHI), y analiza estadísticas descriptivas de los rendimientos: media, varianza, semivarianza, razón de Sharpe y el test de Variance Ratio de Lo-MacKinlay.

In [1]:
#| label: setup
#| code-fold: true
#| code-summary: "Librerías y configuración"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

TICKERS = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META',
    'JPM',  'BAC',  'GS',   'WFC',   'MS',
    'JNJ',  'PFE',  'UNH',  'MRK',   'ABBV',
    'XOM',  'CVX',  'KO',   'PG',    'WMT'
]

START = '2015-01-01'
END   = '2024-12-31'
RF_ANNUAL = 0.0448  # US 10Y Treasury (2024 promedio)
RF_MONTHLY = RF_ANNUAL / 12

print(f"Universo: {len(TICKERS)} acciones del S&P 500")
print(f"Período: {START} → {END}")
print(f"Tasa libre de riesgo mensual: {RF_MONTHLY:.4%}")

Universo: 20 acciones del S&P 500
Período: 2015-01-01 → 2024-12-31
Tasa libre de riesgo mensual: 0.3733%


### Descarga de datos y rendimientos

Descargamos precios ajustados (dividendos y splits) desde Yahoo Finance. El rendimiento mensual se calcula como la variación porcentual del precio ajustado de cierre:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

In [2]:
#| label: descarga
#| code-fold: false

raw = yf.download(TICKERS, start=START, end=END, interval='1mo',
                  auto_adjust=True, progress=False)['Close']

prices = raw.dropna(axis=1, thresh=int(0.9 * len(raw)))
returns = prices.pct_change().dropna()

print(f"Observaciones mensuales: {len(returns)}")
print(f"Acciones disponibles: {returns.shape[1]}")
print(f"\nPrimeros rendimientos:")
returns.head(3).round(4)

Observaciones mensuales: 119
Acciones disponibles: 20

Primeros rendimientos:


Ticker,AAPL,ABBV,AMZN,BAC,CVX,GOOGL,GS,JNJ,JPM,KO,META,MRK,MS,MSFT,PFE,PG,UNH,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,
2015-02-01,0.0964,0.0100,0.0723,0.0436,0.0405,0.0467,0.1008,0.0237,0.1341,0.0517,0.0403,-0.0289,0.0616,0.0854,0.0982,0.0171,0.0695,0.0553,-0.0124,0.0128
2015-03-01,-0.0275,-0.0324,-0.0212,-0.0266,-0.0063,-0.0141,-0.0065,-0.0118,-0.0114,-0.0635,0.0412,-0.0181,-0.0028,-0.0661,0.0226,-0.0375,0.0410,-0.0006,-0.0200,-0.0328
2015-04-01,0.0058,0.1045,0.1335,0.0383,0.0579,-0.0107,0.0450,-0.0139,0.0442,0.0085,-0.0420,0.0446,0.0454,0.1963,-0.0247,-0.0297,-0.0551,0.0129,-0.0454,0.0279


### Índice ponderado por capitalización

El índice replica la lógica de un ETF como el SPY: cada acción pondera según su capitalización relativa al total del portafolio. La base se fija en 100 al inicio del período.

$$\text{Índice}_t = 100 \cdot \prod_{s=1}^{t}\left(1 + \sum_j w_j \cdot r_{j,s}\right)$$

donde $w_j$ es el peso de capitalización de la acción $j$ en el período $s$.

In [3]:
#| label: index-cap
#| code-fold: true

# Capitalización: aproximada con precio × shares out del último período
last_prices = prices.iloc[-1]
shares = {}
for t in returns.columns:
    try:
        info = yf.Ticker(t).fast_info
        shares[t] = info.shares if hasattr(info, 'shares') and info.shares else 1e9
    except:
        shares[t] = 1e9

market_caps = pd.Series({t: last_prices[t] * shares[t] for t in returns.columns})
weights = market_caps / market_caps.sum()

# Índice
port_returns = (returns * weights).sum(axis=1)
index_level = 100 * (1 + port_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(index_level.index, index_level.values, color='#2563eb', linewidth=2)
ax.set_title('Índice ponderado por capitalización (base = 100, enero 2015)', fontsize=13, fontweight='bold')
ax.set_ylabel('Valor del índice')
plt.tight_layout()
plt.show()

print(f"\nRetorno total del período: {(index_level.iloc[-1]/100 - 1):.1%}")
print(f"Valor final del índice: {index_level.iloc[-1]:.1f}")


Retorno total del período: 757.6%
Valor final del índice: 857.6


### Índice Herfindahl-Hirschman (HHI)

El HHI mide la concentración del portafolio. Un portafolio perfectamente diversificado tiene $\text{HHI} = 1/N$; uno completamente concentrado, $\text{HHI} = 1$. El número efectivo de empresas es $n^* = 1/\text{HHI}$.

$$\text{HHI} = \sum_{j=1}^{N} w_j^2 \qquad n^* = \frac{1}{\text{HHI}}$$

In [4]:
#| label: hhi
#| code-fold: true

hhi = (weights ** 2).sum()
n_effective = 1 / hhi
n_total = len(weights)

# Top 5 pesos
top5 = weights.sort_values(ascending=False).head(5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de pesos
top10 = weights.sort_values(ascending=False).head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color='#2563eb', alpha=0.8)
axes[0].set_title('Top 10 pesos en el portafolio', fontweight='bold')
axes[0].set_xlabel('Peso relativo')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

# HHI vs equi-ponderado
labels = ['Portafolio\ncap-weighted', 'Portafolio\nequiponderado']
hhi_vals = [hhi, 1/n_total]
colors = ['#2563eb', '#93c5fd']
axes[1].bar(labels, hhi_vals, color=colors, width=0.4)
axes[1].set_title('HHI: concentración del portafolio', fontweight='bold')
axes[1].set_ylabel('HHI')
for i, v in enumerate(hhi_vals):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"HHI del portafolio cap-weighted: {hhi:.4f}")
print(f"Empresas efectivas (n*):          {n_effective:.1f} de {n_total}")
print(f"\nTop 5 pesos:")
print(top5.apply(lambda x: f'{x:.2%}').to_string())

HHI del portafolio cap-weighted: 0.1193
Empresas efectivas (n*):          8.4 de 20

Top 5 pesos:
AAPL     20.48%
MSFT     17.34%
AMZN     13.23%
GOOGL    12.86%
META      8.28%


### Estadísticas descriptivas y razón de Sharpe

La **razón de Sharpe** mide el retorno en exceso por unidad de riesgo asumido:

$$SR_j = \frac{\bar{r}_j - R_f}{\sigma_j}$$

La **semivarianza** penaliza solo los rendimientos por debajo de la media, siendo una medida de riesgo a la baja más alineada con la percepción del inversionista.

$$SV_j = \frac{1}{T}\sum_{t: r_{jt} < \bar{r}_j}(r_{jt} - \bar{r}_j)^2$$

In [5]:
#| label: descriptiva
#| code-fold: true

desc = pd.DataFrame({
    'Media (%)':      returns.mean() * 100,
    'Mediana (%)':    returns.median() * 100,
    'Std (%)':        returns.std() * 100,
    'Semivarianza':   returns.apply(lambda r: ((r[r < r.mean()] - r.mean())**2).mean()),
    'Sharpe':         (returns.mean() - RF_MONTHLY) / returns.std(),
    'Skewness':       returns.skew(),
    'Kurtosis':       returns.kurt(),
}).round(4)

# Añadir índice de portafolio
pr = port_returns
desc.loc['PORTAFOLIO'] = [
    pr.mean()*100, pr.median()*100, pr.std()*100,
    ((pr[pr < pr.mean()] - pr.mean())**2).mean(),
    (pr.mean() - RF_MONTHLY) / pr.std(),
    pr.skew(), pr.kurt()
]

best_sharpe = desc['Sharpe'].drop('PORTAFOLIO').idxmax()
print(f"Acción con mayor Sharpe: {best_sharpe} ({desc.loc[best_sharpe, 'Sharpe']:.4f})")
print(f"Sharpe del portafolio:   {desc.loc['PORTAFOLIO', 'Sharpe']:.4f}")
print(f"\nEl portafolio supera en Sharpe a {(desc.drop('PORTAFOLIO')['Sharpe'] < desc.loc['PORTAFOLIO', 'Sharpe']).sum()} de {len(TICKERS)} acciones individuales.")
print("→ Consistente con la teoría: la diversificación mejora el Sharpe.")
print("\nEstadísticas completas:")
desc.sort_values('Sharpe', ascending=False)

Acción con mayor Sharpe: MSFT (0.3131)
Sharpe del portafolio:   0.3090

El portafolio supera en Sharpe a 19 de 20 acciones individuales.
→ Consistente con la teoría: la diversificación mejora el Sharpe.

Estadísticas completas:


,Media (%),Mediana (%),Std (%),Semivarianza,Sharpe,Skewness,Kurtosis
Ticker,,,,,,,
MSFT,2.304200,2.371000,6.167400,0.003600,0.313100,0.24040,0.132900
PORTAFOLIO,1.950338,2.544212,5.103128,0.003177,0.309027,-0.25177,-0.005828
AMZN,2.515300,2.546800,8.857200,0.007300,0.241800,0.25040,0.936500
AAPL,2.229600,2.162300,7.969700,0.006400,0.232900,-0.06370,-0.430400
GOOGL,1.890200,1.875300,6.919300,0.004700,0.219200,0.01350,0.159400
UNH,1.619800,1.511300,5.974600,0.003200,0.208600,0.22540,0.643000
JPM,1.734700,2.442000,7.041200,0.005400,0.193300,-0.11660,0.949200
META,2.215000,1.758900,9.761100,0.008900,0.188700,-0.27700,1.997000
WMT,1.292500,1.486100,5.397100,0.003200,0.170300,-0.35670,0.719800


### Test de Variance Ratio (Lo & MacKinlay, 1988)

El test compara la varianza de rendimientos anuales con la varianza mensual anualizada. Bajo la hipótesis de caminata aleatoria (mercados eficientes), ambas deberían ser iguales: $VR = 1$.

- $VR < 1$: autocorrelación negativa (reversión a la media)
- $VR > 1$: autocorrelación positiva (momentum)

$$VR = \frac{\hat{\sigma}^2_{\text{anual}}}{12 \cdot \hat{\sigma}^2_{\text{mensual}}}$$

In [6]:
#| label: variance-ratio
#| code-fold: true

# Rendimientos anuales del portafolio (interanual)
index_monthly = index_level.resample('YE').last()
annual_returns = index_monthly.pct_change().dropna()

var_annual  = annual_returns.var()
var_monthly = port_returns.var()
var_monthly_annualized = var_monthly * 12

VR = var_annual / var_monthly_annualized

# Z-stat aproximado (Lo-MacKinlay)
T = len(port_returns)
q = 12
z_stat = (VR - 1) / np.sqrt(2 * (2*q - 1) * (q - 1) / (3*q*T))
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("=" * 45)
print("Test de Variance Ratio — Portafolio")
print("=" * 45)
print(f"Varianza anual observada:        {var_annual:.6f}")
print(f"Varianza mensual × 12:           {var_monthly_annualized:.6f}")
print(f"\nVR = {VR:.4f}")
print(f"Z-stat: {z_stat:.4f}  |  p-value: {p_value:.4f}")
print()
if p_value < 0.05:
    direction = "negativa (reversión a la media)" if VR < 1 else "positiva (momentum)"
    print(f"→ Se rechaza caminata aleatoria (p < 0.05).")
    print(f"  Hay autocorrelación {direction}.")
else:
    print(f"→ No se rechaza caminata aleatoria (p ≥ 0.05).")

Test de Variance Ratio — Portafolio
Varianza anual observada:        0.061907
Varianza mensual × 12:           0.031250

VR = 1.9810
Z-stat: 2.8544  |  p-value: 0.0043

→ Se rechaza caminata aleatoria (p < 0.05).
  Hay autocorrelación positiva (momentum).
